# 04 - Avaliação e Inferência

## 🎯 Objetivo

Comparar KNN e SVD, selecionar o modelo final, exportá-lo e demonstrar a função de inferência.

## 1. Setup

In [ ]:
import os
import pathlib
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict
from sklearn.model_selection import train_test_split as skl_split
from surprise import SVD, KNNWithMeans, Dataset, Reader, accuracy, dump as surprise_dump

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_theme(style="whitegrid")

try:
    _HERE = pathlib.Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    _HERE = pathlib.Path(os.path.abspath("")).resolve()

DATA_PATH  = _HERE.parent / "dataset" / "dataset.csv"
MODELS_DIR = _HERE.parent / "models"

# Reproduz o mesmo split 80/20 de 03_svd.ipynb (random_state=42 garante identidade)
df = pd.read_csv(DATA_PATH)
df_train, df_test = skl_split(df, test_size=0.20, random_state=42)

reader     = Reader(rating_scale=(1, 5))
data_train = Dataset.load_from_df(df_train[["user_id", "item_id", "ranking"]], reader)
trainset   = data_train.build_full_trainset()
testset    = list(zip(df_test["user_id"], df_test["item_id"], df_test["ranking"]))
todos_itens = set(df["item_id"].unique())

print(f"Dataset carregado: {df.shape[0]:,} interações")
print(f"Treino: {len(df_train):,} | Teste: {len(df_test):,}")

## 2. Métricas de Avaliação

### 2.1. Métricas de regressão

**RMSE (Root Mean Squared Error)**

$$\text{RMSE} = \sqrt{\frac{1}{|\mathcal{T}|} \sum_{(u,i) \in \mathcal{T}} (r_{ui} - \hat{r}_{ui})^2}$$

Penaliza erros grandes mais fortemente.

**MAE (Mean Absolute Error)**

$$\text{MAE} = \frac{1}{|\mathcal{T}|} \sum_{(u,i) \in \mathcal{T}} |r_{ui} - \hat{r}_{ui}|$$

Mais interpretável e menos sensível a outliers.

### 2.2. Métricas de classificação (top-N)

Tratando recomendação como classificação binária com threshold (ex: rating ≥ 4 = relevante):

- **Precisão@K:** dos K recomendados, quantos são relevantes
- **Recall@K:** dos relevantes, quantos foram recomendados no top-K
- **F1@K:** média harmônica entre precisão e recall
- **Acurácia:** proporção de classificações corretas

> ✏️ **TODO:** definir threshold e justificar.

In [ ]:
# ── Treina modelos de avaliação (80% dos dados) ──────────────────────────────

modelo_svd_eval = SVD(n_factors=20, lr_all=0.002, reg_all=0.1, n_epochs=50,
                      random_state=42, verbose=False)
modelo_svd_eval.fit(trainset)

sim_options = {"name": "cosine", "user_based": False}
modelo_knn_eval = KNNWithMeans(k=5, sim_options=sim_options, verbose=False)
modelo_knn_eval.fit(trainset)

# ── Predições no conjunto de teste ───────────────────────────────────────────
pred_svd = modelo_svd_eval.test(testset)
pred_knn = modelo_knn_eval.test(testset)

# ── Baseline ingênuo: prediz sempre a média global ───────────────────────────
media_global = df_train["ranking"].mean()
pred_naive   = [(uid, iid, r, media_global, {}) for uid, iid, r in testset]

# ── Métricas de regressão ────────────────────────────────────────────────────
rmse_naive = accuracy.rmse(pred_naive, verbose=False)
rmse_svd   = accuracy.rmse(pred_svd,   verbose=False)
rmse_knn   = accuracy.rmse(pred_knn,   verbose=False)

mae_svd = accuracy.mae(pred_svd, verbose=False)
mae_knn = accuracy.mae(pred_knn, verbose=False)

print("=== Métricas de Regressão (escala 1–5) ===")
print(f"Baseline ingênuo (média global) — RMSE: {rmse_naive:.4f}")
print(f"SVD                            — RMSE: {rmse_svd:.4f}  |  MAE: {mae_svd:.4f}")
print(f"KNN (KNNWithMeans, k=5)        — RMSE: {rmse_knn:.4f}  |  MAE: {mae_knn:.4f}")
print()
print("Nota: ambos os modelos estão próximos do baseline ingênuo em RMSE.")
print("Isso reflete o feedback implícito (94% das interações com ranking=1).")
print("→ Use Hit Rate@K e NDCG@K abaixo para medir qualidade de recomendação.")

In [ ]:
from collections import defaultdict
import time

# ── Ground truth ──────────────────────────────────────────────────────────────
# Usuários elegíveis: têm interações no TREINO e no TESTE
train_items_per_user = defaultdict(set)
for _, row in df_train.iterrows():
    train_items_per_user[row["user_id"]].add(row["item_id"])

test_items_per_user = defaultdict(set)
for _, row in df_test.iterrows():
    test_items_per_user[row["user_id"]].add(row["item_id"])

eligible_users = [u for u in test_items_per_user if train_items_per_user[u]]
print(f"Usuários elegíveis (treino+teste): {len(eligible_users):,}")

# ── Avaliação CORRETA de ranking ───────────────────────────────────────────────
# NOTA: não se deve usar pred_svd.test(testset) para calcular Hit Rate@K,
# pois isso gera recomendações APENAS dos itens do testset, garantindo
# acerto trivial (100%). A avaliação correta prediz TODOS os itens não vistos.

K = 10
ts = modelo_svd_eval.trainset

hits_svd = hits_knn = 0
ndcg_svd_scores, ndcg_knn_scores = [], []
covered_svd, covered_knn = set(), set()

# ── SVD: vetorizado sobre todos os usuários elegíveis ─────────────────────────
print(f"SVD: avaliando {len(eligible_users):,} usuários (vetorizado)...")
t0 = time.time()
for uid in eligible_users:
    consumidos = train_items_per_user[uid]
    relevantes = test_items_per_user[uid]
    try:
        inner_uid = ts.to_inner_uid(uid)
    except ValueError:
        continue
    all_scores = (
        ts.global_mean + modelo_svd_eval.bu[inner_uid]
        + modelo_svd_eval.bi + modelo_svd_eval.qi @ modelo_svd_eval.pu[inner_uid]
    ).clip(1, 5)
    candidatos = sorted(
        [(ts.to_raw_iid(i), float(all_scores[i]))
         for i in range(ts.n_items) if ts.to_raw_iid(i) not in consumidos],
        key=lambda x: x[1], reverse=True
    )
    top_k = [item for item, _ in candidatos[:K]]
    if set(top_k) & relevantes:
        hits_svd += 1
    gains = [1 if item in relevantes else 0 for item in top_k]
    dcg  = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(sorted(gains, reverse=True)))
    ndcg_svd_scores.append(dcg / idcg if idcg > 0 else 0.0)
    covered_svd.update(top_k)
print(f"  Tempo: {time.time()-t0:.1f}s")

# ── KNN: avaliação em amostra (individual predict é O(n_itens) por usuário) ───
N_SAMPLE = 500
np.random.seed(42)
sample_users = np.random.choice(eligible_users, min(N_SAMPLE, len(eligible_users)), replace=False)
print(f"KNN: avaliando amostra de {len(sample_users)} usuários...")
t0 = time.time()
for uid in sample_users:
    consumidos = train_items_per_user[uid]
    relevantes = test_items_per_user[uid]
    nao_vistos = list(todos_itens - consumidos)
    preds = [(item, modelo_knn_eval.predict(uid, item).est) for item in nao_vistos]
    preds.sort(key=lambda x: x[1], reverse=True)
    top_k = [item for item, _ in preds[:K]]
    if set(top_k) & relevantes:
        hits_knn += 1
    gains = [1 if item in relevantes else 0 for item in top_k]
    dcg  = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(sorted(gains, reverse=True)))
    ndcg_knn_scores.append(dcg / idcg if idcg > 0 else 0.0)
    covered_knn.update(top_k)
print(f"  Tempo: {time.time()-t0:.1f}s")

hr_svd  = hits_svd / len(eligible_users)
hr_knn  = hits_knn / len(sample_users)
nd_svd  = float(np.mean(ndcg_svd_scores))
nd_knn  = float(np.mean(ndcg_knn_scores))
cov_svd = len(covered_svd) / len(todos_itens)
cov_knn = len(covered_knn) / len(todos_itens)

print(f"\n=== Métricas de Ranking (K={K}) ===")
print(f"{'Métrica':<30} {'SVD (n={:,})'.format(len(eligible_users)):>14} {'KNN (n={})'.format(len(sample_users)):>14}")
print("-" * 60)
print(f"{'Hit Rate@K':<30} {hr_svd:>14.4f} {hr_knn:>14.4f}")
print(f"{'NDCG@K':<30} {nd_svd:>14.4f} {nd_knn:>14.4f}")
print(f"{'Cobertura Catálogo@K':<30} {cov_svd:>14.4f} {cov_knn:>14.4f}")
print()
print("Interpretação:")
print(f"  Chance aleatória de acerto: ~{K/len(todos_itens)*100:.1f}% (10/{len(todos_itens)} itens)")
print("  SVD: alta precisão, baixa cobertura (viés de popularidade)")
print("  KNN: maior diversidade de catálogo")

## 3. Comparação KNN vs SVD

In [ ]:
hr_svd  = hit_rate_at_k(top_n_svd, test_items_per_user)
hr_knn  = hit_rate_at_k(top_n_knn, test_items_per_user)
ndcg_svd = ndcg_at_k(top_n_svd, test_items_per_user, K)
ndcg_knn = ndcg_at_k(top_n_knn, test_items_per_user, K)
cov_svd  = catalog_coverage(top_n_svd, todos_itens)
cov_knn  = catalog_coverage(top_n_knn, todos_itens)

comparacao = pd.DataFrame({
    "Modelo":              ["Baseline (média global)", "SVD (Funk SVD)",    "KNN (KNNWithMeans, k=5)"],
    "RMSE ↓":             [round(rmse_naive, 4),      round(rmse_svd, 4),  round(rmse_knn, 4)],
    "MAE ↓":              ["-",                        round(mae_svd, 4),   round(mae_knn, 4)],
    f"Hit Rate@{K} ↑":    ["-",                        round(hr_svd, 4),    round(hr_knn, 4)],
    f"NDCG@{K} ↑":        ["-",                        round(ndcg_svd, 4),  round(ndcg_knn, 4)],
    f"Cobertura@{K} ↑":   ["-",                        round(cov_svd, 4),   round(cov_knn, 4)],
})

print("=== Tabela Comparativa KNN vs SVD ===")
print(comparacao.to_string(index=False))

# ── Visualização: Hit Rate e NDCG lado a lado ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
modelos = ["SVD", "KNN"]
cores   = ["#4C78A8", "#F28E2B"]

axes[0].bar(modelos, [hr_svd, hr_knn], color=cores, edgecolor="white")
axes[0].set_title(f"Hit Rate@{K}")
axes[0].set_ylabel("Proporção de usuários com acerto")
axes[0].set_ylim(0, max(hr_svd, hr_knn) * 1.4)
for i, v in enumerate([hr_svd, hr_knn]):
    axes[0].text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=11)

axes[1].bar(modelos, [ndcg_svd, ndcg_knn], color=cores, edgecolor="white")
axes[1].set_title(f"NDCG@{K}")
axes[1].set_ylabel("Score médio (0–1)")
axes[1].set_ylim(0, max(ndcg_svd, ndcg_knn) * 1.4)
for i, v in enumerate([ndcg_svd, ndcg_knn]):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=11)

plt.suptitle("Comparação de Modelos — Métricas de Ranking", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Seleção do Modelo Final

In [ ]:
# Seleciona automaticamente o modelo com maior Hit Rate@K
melhor_modelo_nome = "SVD" if hr_svd >= hr_knn else "KNN"
melhor_modelo_eval = modelo_svd_eval if hr_svd >= hr_knn else modelo_knn_eval

print(f"Modelo selecionado: {melhor_modelo_nome}")
print()
print("Critério de seleção: Hit Rate@10 (métrica primária para feedback implícito)")
print()
print("Justificativa:")
print("  • RMSE/MAE não distinguem bem os modelos em feedback implícito (ambos")
print("    próximos do baseline ingênuo de média global).")
print("  • Hit Rate@10 mede diretamente o que importa: o modelo colocou ao menos")
print("    um item relevante entre os 10 mais recomendados?")
print("  • NDCG@10 complementa penalizando itens relevantes em posições inferiores.")
print()
print(f"  SVD  → Hit Rate@10={hr_svd:.4f}, NDCG@10={ndcg_svd:.4f}, Cobertura={cov_svd:.4f}")
print(f"  KNN  → Hit Rate@10={hr_knn:.4f}, NDCG@10={ndcg_knn:.4f}, Cobertura={cov_knn:.4f}")

## 5. Exportação do Modelo

In [ ]:
import sys
sys.path.insert(0, str(MODELS_DIR))
from inference import recomendar

# Usuário com histórico (warm-start)
usuario_warm = int(df.groupby("user_id").size().sort_values(ascending=False).index[0])
historico = df[df["user_id"] == usuario_warm][["item_id", "ranking"]]
print(f"=== Usuário {usuario_warm} (warm-start) — {len(historico)} interações ===")
print(historico.to_string(index=False))
print("\nTop-5 recomendações:")
for rec in recomendar(usuario_warm, n=5):
    print(f"  {rec['espetaculo'][:60]:<60}  score={rec['score']}")

print()

# Usuário inexistente (cold-start)
print("=== Usuário 999999999 (cold-start) ===")
print("Top-5 por popularidade (score=None = sem histórico):")
for rec in recomendar(999999999, n=5):
    print(f"  {rec['espetaculo'][:60]:<60}  score={rec['score']}")

## 6. Função de Inferência

A função está implementada em `src/models/inference.py`.

> ✏️ **TODO:** documentar parâmetros, retorno e exceções.

## 7. Demonstração

> ✏️ **TODO:** exemplo de uso gerando recomendações para usuário(s).

## 8. Conclusões